# Advanced SQL Business Analysis

## Project Setup

In this notebook, we reconnect to the SQLite database and import the required libraries before performing advanced SQL analysis.

In [37]:
import pandas as pd
import sqlite3

connection = sqlite3.connect("global_electronics.db")

In [38]:
import os

print(os.getcwd())

c:\Users\dhine\OneDrive\Documents\GLOBAL ELECTRONICS RETAIL BUSINESS INTELLIGENCE DASHBOARD\sql


In [39]:
import os

print(os.listdir())

['01_Create_Database.ipynb', '02_Business_Queries.sql', '03_Run_Business_Queries.ipynb', '04_Advanced_SQL_Analysis.ipynb', 'global_electronics.db', 'global_electronics_retail.db']


In [40]:
import os

for file in os.listdir():
    print(file)

01_Create_Database.ipynb
02_Business_Queries.sql
03_Run_Business_Queries.ipynb
04_Advanced_SQL_Analysis.ipynb
global_electronics.db
global_electronics_retail.db


In [41]:
import pandas as pd
import sqlite3

connection = sqlite3.connect("global_electronics.db")

pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    connection
)

,name
0,customers
1,products
2,sales
3,stores
4,exchange_rates


# 1. LEFT JOIN Analysis

## Objective

LEFT JOIN allows us to keep all records from one table even if matching records do not exist in another table.

This is useful for identifying missing relationships and performing complete business reporting.

## Business Question 1

### Which products have never been sold?

This analysis identifies products that have never appeared in the sales table.

Business Value:
- Detect slow-moving inventory.
- Identify products requiring marketing campaigns.
- Remove obsolete products from inventory.

In [42]:
query = """
SELECT
    p."Product Name",
    p.Brand,
    p.Category
FROM products p

LEFT JOIN sales s
ON p.ProductKey = s.ProductKey

WHERE s.ProductKey IS NULL;
"""

pd.read_sql(query, connection)

,Product Name,Brand,Category
0,Contoso Home Theater System 5.1 Channel M1530 ...,Contoso,TV and Video
1,Contoso Washer & Dryer 21in E210 Green,Contoso,Home Appliances
2,Adventure Works Floor Lamp X1150 Black,Adventure Works,Home Appliances
3,Adventure Works Floor Lamp M2150 Black,Adventure Works,Home Appliances
4,Adventure Works Chandelier M6150 Black,Adventure Works,Home Appliances
5,Adventure Works Wall Lamp E3150 Silver,Adventure Works,Home Appliances
6,Adventure Works Desk Lamp E1300 Silver,Adventure Works,Home Appliances
7,Adventure Works Floor Lamp X1150 Grey,Adventure Works,Home Appliances
8,Adventure Works Wall Lamp E3150 Grey,Adventure Works,Home Appliances
9,Adventure Works Desk Lamp E1300 Grey,Adventure Works,Home Appliances


## Business Question 2

### Which stores have not processed any sales?

This query identifies stores that have never recorded a sale.

Business Value:
- Detect underperforming stores.
- Evaluate store operations.
- Support business expansion decisions.

In [43]:
query = """
SELECT
    st.StoreKey,
    st.Country,
    st.State

FROM stores st

LEFT JOIN sales s
ON st.StoreKey = s.StoreKey

WHERE s.StoreKey IS NULL;
"""

pd.read_sql(query, connection)

,StoreKey,Country,State
0,3,Australia,South Australia
1,7,Canada,New Brunswick
2,11,Canada,Yukon
3,25,Germany,Mecklenburg-Vorpommern
4,35,Netherlands,Zeeland
5,46,United States,Delaware
6,52,United States,Mississippi
7,58,United States,North Dakota
8,60,United States,Rhode Island


### Business Insight

LEFT JOIN helps identify missing relationships between tables.

This is commonly used for:

- Unsold products
- Inactive customers
- Stores without transactions
- Missing business records

# 2. CASE WHEN Analysis

## Objective

The CASE WHEN statement allows us to classify data into business-friendly categories.

Business analysts frequently use CASE statements to create customer segments, product classifications, and sales performance categories.

This improves reporting and supports better business decision-making.

## Business Question 3

### Classify Products by Price Range

Business Value:

- Identify premium products.
- Identify mid-range products.
- Identify budget products.
- Assist pricing strategy and inventory planning.

In [44]:
query = """
SELECT
    "Product Name",
    Brand,
    Category,
    "Unit Price USD",

    CASE
        WHEN CAST(REPLACE("Unit Price USD",'$','') AS REAL) >= 500
            THEN 'Premium'

        WHEN CAST(REPLACE("Unit Price USD",'$','') AS REAL) >= 100
            THEN 'Mid Range'

        ELSE 'Budget'
    END AS Price_Category

FROM products;
"""

pd.read_sql(query, connection)

,Product Name,Brand,Category,Unit Price USD,Price_Category
0,Contoso 512MB MP3 Player E51 Silver,Contoso,Audio,$12.99,Budget
1,Contoso 512MB MP3 Player E51 Blue,Contoso,Audio,$12.99,Budget
2,Contoso 1G MP3 Player E100 White,Contoso,Audio,$14.52,Budget
3,Contoso 2G MP3 Player E200 Silver,Contoso,Audio,$21.57,Budget
4,Contoso 2G MP3 Player E200 Red,Contoso,Audio,$21.57,Budget
...,...,...,...,...,...
2512,Contoso Bluetooth Active Headphones L15 Red,Contoso,Cell phones,$129.99,Mid Range
2513,Contoso Bluetooth Active Headphones L15 White,Contoso,Cell phones,$129.99,Mid Range
2514,Contoso In-Line Coupler E180 White,Contoso,Cell phones,$3.35,Budget
2515,Contoso In-Line Coupler E180 Black,Contoso,Cell phones,$3.35,Budget


### Business Insight

CASE WHEN helps classify products into meaningful pricing segments.

This enables management to:

- Design pricing strategies.
- Compare premium and budget product performance.
- Improve product positioning.

## Business Question 4

### Count Products in Each Price Segment

In [45]:
query = """
SELECT

CASE
    WHEN CAST(REPLACE("Unit Price USD",'$','') AS REAL) >= 500
        THEN 'Premium'

    WHEN CAST(REPLACE("Unit Price USD",'$','') AS REAL) >= 100
        THEN 'Mid Range'

    ELSE 'Budget'
END AS Price_Category,

COUNT(*) AS Total_Products

FROM products

GROUP BY Price_Category

ORDER BY Total_Products DESC;
"""

pd.read_sql(query, connection)

,Price_Category,Total_Products
0,Mid Range,1223
1,Budget,947
2,Premium,347


### Business Insight

This report summarizes the number of products available in each pricing segment.

It provides valuable insights into the company's product portfolio and pricing distribution.

# 3. HAVING Clause Analysis

## Objective

The HAVING clause filters grouped records after aggregation.

Unlike the WHERE clause, HAVING works with aggregate functions such as SUM(), COUNT(), and AVG().

Business analysts use HAVING to identify high-performing products, brands, stores, and regions.

## Business Question 5

### Which brands have sold more than 10,000 units?

Business Value:

- Identify top-performing brands.
- Support marketing and inventory decisions.
- Understand brand contribution to overall sales.

In [46]:
query = """
SELECT
    p.Brand,
    SUM(s.Quantity) AS Total_Units_Sold

FROM sales s

INNER JOIN products p
ON s.ProductKey = p.ProductKey

GROUP BY p.Brand

HAVING SUM(s.Quantity) > 10000

ORDER BY Total_Units_Sold DESC;
"""

pd.read_sql(query, connection)

,Brand,Total_Units_Sold
0,Contoso,49827
1,Wide World Importers,27413
2,Southridge Video,24814
3,Adventure Works,20099
4,The Phone Company,18764
5,Tailspin Toys,17455
6,Fabrikam,11384


### Business Insight

The HAVING clause helps identify brands that contribute significantly to total sales volume.

These brands may deserve increased marketing investment, inventory allocation, and strategic focus.

## Business Question 6

### Which product categories sold more than 20,000 units?

Business Value:

- Identify the strongest product categories.
- Understand customer demand.
- Improve product portfolio planning.

In [47]:
query = """
SELECT
    p.Category,
    SUM(s.Quantity) AS Total_Units_Sold

FROM sales s

INNER JOIN products p
ON s.ProductKey = p.ProductKey

GROUP BY p.Category

HAVING SUM(s.Quantity) > 20000

ORDER BY Total_Units_Sold DESC;
"""

pd.read_sql(query, connection)

,Category,Total_Units_Sold
0,Computers,44151
1,Cell phones,31477
2,"Music, Movies and Audio Books",28802
3,Audio,23490
4,Games and Toys,22591


### Business Insight

High-performing product categories contribute the largest share of sales volume.

These categories should receive priority for promotions, inventory replenishment, and future product expansion.

# 4. Common Table Expressions (CTE)

## Objective

A Common Table Expression (CTE) creates a temporary named result set that can be referenced within a SQL query.

CTEs improve query readability, simplify complex business reports, and eliminate repeated calculations.

They are widely used in Business Intelligence, Data Warehousing, and Financial Analytics.

## Business Question 7

### Which brands perform above the average sales quantity?

Business Value

- Compare each brand with the company average.
- Identify high-performing brands.
- Support strategic marketing decisions.

In [48]:
query = """
WITH BrandSales AS
(
    SELECT
        p.Brand,
        SUM(s.Quantity) AS Total_Units

    FROM sales s

    INNER JOIN products p
    ON s.ProductKey = p.ProductKey

    GROUP BY p.Brand
)

SELECT
    Brand,
    Total_Units

FROM BrandSales

WHERE Total_Units >
(
    SELECT AVG(Total_Units)
    FROM BrandSales
)

ORDER BY Total_Units DESC;
"""

pd.read_sql(query, connection)

,Brand,Total_Units
0,Contoso,49827
1,Wide World Importers,27413
2,Southridge Video,24814
3,Adventure Works,20099
4,The Phone Company,18764


### Business Insight

Using a CTE makes complex analytical queries easier to understand and maintain.

Brands performing above the company average represent strong business opportunities for marketing investment and inventory planning.

## Business Question 8

### Rank brands based on total sales quantity

Business Value

- Compare brand performance.
- Identify market leaders.
- Build executive ranking reports.

In [49]:
query = """
WITH BrandSales AS
(
    SELECT

        p.Brand,

        SUM(s.Quantity) AS Total_Units

    FROM sales s

    INNER JOIN products p

    ON s.ProductKey = p.ProductKey

    GROUP BY p.Brand
)

SELECT

    Brand,

    Total_Units,

    RANK() OVER
    (
        ORDER BY Total_Units DESC
    ) AS Brand_Rank

FROM BrandSales;
"""

pd.read_sql(query, connection)

,Brand,Total_Units,Brand_Rank
0,Contoso,49827,1
1,Wide World Importers,27413,2
2,Southridge Video,24814,3
3,Adventure Works,20099,4
4,The Phone Company,18764,5
5,Tailspin Toys,17455,6
6,Fabrikam,11384,7
7,Proseware,9427,8
8,Northwind Traders,7610,9
9,A. Datum,5655,10


### Business Insight

Ranking brands enables management to benchmark performance and identify the strongest contributors to overall sales.

# 5. Window Functions

## Objective

Window functions perform calculations across rows without collapsing the dataset.

Functions covered:

- ROW_NUMBER()
- RANK()
- DENSE_RANK()

These are widely used in financial reporting, sales analytics, and business intelligence dashboards.

## Business Question 9

### Rank products within each category

Business Value

- Identify the best-selling products.
- Compare products within the same category.
- Support inventory optimization.

In [50]:
query = """
SELECT

    p.Category,

    p."Product Name",

    SUM(s.Quantity) AS Total_Units,

    ROW_NUMBER() OVER
    (
        PARTITION BY p.Category

        ORDER BY SUM(s.Quantity) DESC
    ) AS Product_Rank

FROM sales s

INNER JOIN products p

ON s.ProductKey = p.ProductKey

GROUP BY

    p.Category,

    p."Product Name";
"""

pd.read_sql(query, connection)

,Category,Product Name,Total_Units,Product_Rank
0,Audio,WWI 1GB Digital Voice Recorder Pen E100 Black,431,1
1,Audio,WWI 1GB Digital Voice Recorder Pen E100 Red,388,2
2,Audio,WWI 4GB Video Recording Pen X200 Yellow,370,3
3,Audio,WWI 4GB Video Recording Pen X200 Black,366,4
4,Audio,WWI Wireless Bluetooth Stereo Headphones M270 ...,359,5
...,...,...,...,...
2487,TV and Video,Contoso Home Theater System 2.1 Channel E1200 ...,3,217
2488,TV and Video,Litware Home Theater System 7.1 Channel M710 B...,2,218
2489,TV and Video,Litware Home Theater System 5.1 Channel M511 B...,2,219
2490,TV and Video,Contoso Home Theater System 5.1 Channel M1530 ...,2,220


### Business Insight

ROW_NUMBER() identifies the highest-performing product within each category, helping business teams prioritize inventory, promotions, and product placement.

# 6. Executive Business Reports

## Objective

This section demonstrates executive-level SQL reports commonly used by management teams to monitor business performance.

These reports support strategic decision-making by highlighting the company's top-performing products, brands, stores, and markets.

## Business Question 10

### Top 10 Best-Selling Products

Business Value

- Identify the company's highest-selling products.
- Support inventory planning.
- Improve marketing campaigns.

In [51]:
query = """
SELECT

    p."Product Name",

    p.Brand,

    p.Category,

    SUM(s.Quantity) AS Total_Units_Sold

FROM sales s

INNER JOIN products p

ON s.ProductKey = p.ProductKey

GROUP BY

    p."Product Name",
    p.Brand,
    p.Category

ORDER BY Total_Units_Sold DESC

LIMIT 10;
"""

pd.read_sql(query, connection)

,Product Name,Brand,Category,Total_Units_Sold
0,WWI Desktop PC2.33 X2330 Black,Wide World Importers,Computers,550
1,WWI Desktop PC1.80 E1800 White,Wide World Importers,Computers,538
2,Adventure Works Desktop PC1.60 ED160 Black,Adventure Works,Computers,521
3,Adventure Works Desktop PC2.30 MD230 White,Adventure Works,Computers,521
4,Adventure Works Desktop PC1.80 ED180 Black,Adventure Works,Computers,520
5,Adventure Works Desktop PC2.30 MD230 Black,Adventure Works,Computers,514
6,WWI Desktop PC1.60 E1600 Black,Wide World Importers,Computers,509
7,WWI Desktop PC1.60 E1600 Silver,Wide World Importers,Computers,507
8,Adventure Works Desktop PC1.60 ED160 White,Adventure Works,Computers,505
9,WWI Desktop PC1.60 E1600 Red,Wide World Importers,Computers,505


### Business Insight

The best-selling products generate the highest demand and should receive priority in inventory management, forecasting, and promotional campaigns.

## Business Question 11

### Top 10 Brands by Sales Quantity

In [52]:
query = """
SELECT

    p.Brand,

    SUM(s.Quantity) AS Total_Units_Sold

FROM sales s

INNER JOIN products p

ON s.ProductKey = p.ProductKey

GROUP BY p.Brand

ORDER BY Total_Units_Sold DESC

LIMIT 10;
"""

pd.read_sql(query, connection)

,Brand,Total_Units_Sold
0,Contoso,49827
1,Wide World Importers,27413
2,Southridge Video,24814
3,Adventure Works,20099
4,The Phone Company,18764
5,Tailspin Toys,17455
6,Fabrikam,11384
7,Proseware,9427
8,Northwind Traders,7610
9,A. Datum,5655


### Business Insight

This report identifies the strongest brands based on sales quantity and helps management focus on high-performing product lines.

## Business Question 12

### Top Countries by Sales Volume

In [53]:
query = """
SELECT

    st.Country,

    SUM(s.Quantity) AS Total_Units_Sold

FROM sales s

INNER JOIN stores st

ON s.StoreKey = st.StoreKey

GROUP BY st.Country

ORDER BY Total_Units_Sold DESC;
"""

pd.read_sql(query, connection)

,Country,Total_Units_Sold
0,United States,83638
1,Online,41311
2,United Kingdom,20625
3,Germany,14880
4,Canada,12991
5,Australia,7085
6,Italy,6986
7,Netherlands,5909
8,France,4332


### Business Insight

Understanding country-level sales performance helps management allocate resources, plan regional strategies, and identify growth opportunities.

## Business Question 13

### Top Categories by Sales Volume

In [54]:
query = """
SELECT

    p.Category,

    SUM(s.Quantity) AS Total_Units_Sold

FROM sales s

INNER JOIN products p

ON s.ProductKey = p.ProductKey

GROUP BY p.Category

ORDER BY Total_Units_Sold DESC;
"""

pd.read_sql(query, connection)

,Category,Total_Units_Sold
0,Computers,44151
1,Cell phones,31477
2,"Music, Movies and Audio Books",28802
3,Audio,23490
4,Games and Toys,22591
5,Home Appliances,18401
6,Cameras and camcorders,17609
7,TV and Video,11236


### Business Insight

Category-level analysis highlights which product groups contribute most to sales and supports product portfolio optimization.

# Advanced SQL Summary

## SQL Concepts Covered

- SELECT
- Aggregate Functions
- GROUP BY
- ORDER BY
- INNER JOIN
- LEFT JOIN
- CASE WHEN
- HAVING
- CTE (WITH)
- ROW_NUMBER()
- RANK()

## Business Reports Developed

- Product Performance
- Brand Performance
- Country Performance
- Category Performance
- Ranking Reports
- Executive Business Reports

## Outcome

This notebook demonstrates advanced SQL techniques and real-world business reporting skills commonly required for Data Analyst, Business Analyst, and Business Intelligence roles.